# 🔬 Druggability 文献检索 Quickstart

本 Notebook 演示完整流程：**搜索 → 下载 → 解析 → 实体抽取**

```bash
conda activate research
jupyter lab
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # 加载 API 配置

## 1. 搜索论文 (OpenAlex)

In [ ]:
from litkit.search import search_openalex
import pandas as pd

results = search_openalex("PROTAC druggability", limit=10)

df = pd.DataFrame([
    {
        'title': r.get('title', ''),
        'year': r.get('publication_year', ''),
        'doi': r.get('doi', ''),
        'cited_by': r.get('cited_by_count', 0),
        'open_access': r.get('open_access', {}).get('is_oa', False),
    }
    for r in results
])

df.sort_values('cited_by', ascending=False)

## 2. 多源搜索对比

In [ ]:
from litkit.search import search

# 同一个关键词，不同数据源
for source in ['openalex', 'crossref', 'arxiv']:
    try:
        hits = search('druggable genome', source=source, limit=3)
        print(f"\n=== {source.upper()} ({len(hits)} hits) ===")
        for h in hits[:3]:
            title = h.get('title', 'N/A')
            print(f"  • {title[:80]}")
    except Exception as e:
        print(f"  {source}: {e}")

## 3. PubMed 搜索

In [ ]:
from litkit.search import search_pubmed

pubmed_results = search_pubmed("PROTAC druggability", limit=5)
for r in pubmed_results:
    print(f"PMID: {r['pmid']} | {r['year']} | {r['title'][:70]}")
    if r['abstract']:
        print(f"  摘要: {r['abstract'][:120]}...")
    print()

## 4. 实体识别 (PubTator3 API)

In [ ]:
from litkit.ner import annotate_with_pubtator, regex_drug_entities

# 取第一篇 PubMed 结果的摘要做 NER
if pubmed_results:
    text = pubmed_results[0]['abstract']
    print(f"文本: {text[:200]}...\n")
    
    entities = annotate_with_pubtator(text)
    if entities:
        ner_df = pd.DataFrame(entities)
        display(ner_df)
    else:
        print("PubTator3 未返回实体，尝试正则:")
        drugs = regex_drug_entities(text)
        print(f"  匹配到的药物名: {drugs}")

## 5. ChEMBL 靶点查询

In [ ]:
from chembl_webresource_client.new_client import new_client

target = new_client.target
# 搜索 EGFR 靶点
egfr_targets = target.search('EGFR')
egfr_df = pd.DataFrame(egfr_targets[:5])
egfr_df[['target_chembl_id', 'pref_name', 'target_type', 'organism']]

## 6. 下一步

- 📥 **下载 PDF**: `from litkit.fetch import download_pdf, fetch_unpaywall_pdf_url`
- 📄 **解析 PDF**: `from litkit.parse import extract_text_pymupdf, parse_with_grobid`
- 🤖 **论文问答**: 安装 `pip install paper-qa` 后配合 LLM 使用
- 🐳 **GROBID**: `sudo docker run -d -p 8070:8070 lfoppiano/grobid:0.8.0`